# OpenAI API con Python

**Conceptos:** hacer llamadas a la API, usar instrucciones, JSON estructurado, y tools

## 1. Setup

Instalar dependencias.

Ejecutar una vez.

In [8]:
%pip install --upgrade openai pydantic


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. API Key

Usar variable de entorno.

No hardcodear secrets.

In [10]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-nano-2026-03-17")

# Importar OpenAI
from openai import OpenAI

# inicializar cliente
client = OpenAI()

print(f"Modelo: {OPENAI_MODEL}")
print("Cliente listo")

Modelo: gpt-5.4-nano-2026-03-17
Cliente listo


## 3. Primera llamada

Caso mínimo: enviar texto y leer respuesta.

In [14]:
# ACLARACION:

response = client.responses.create(
    model=OPENAI_MODEL, # gpt-5.4-nano
    input="Explica qué es una API en una frase simple.",
)

print(response)
print(response.output_text)

Response(id='resp_0b1da4b179ad59c5006a07ac07196c81908d0159006ff9ddbd', created_at=1778887687.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-nano-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_0b1da4b179ad59c5006a07ac078d608190a9661cfe021e6508', content=[ResponseOutputText(annotations=[], text='Una API es un intermediario que permite que un programa se comunique con otro para pedir y recibir datos o funciones.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=1778887687.0, conversation=None, max_output_tokens=None, max_tool_calls=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_retention='in_memory', reasoning=Reasoning(effort='none', generate_summary=None, summary=None), safety_identifier=None, service_tier='defau

## 4. Instrucciones

`instructions` define rol, tono y formato.

In [ ]:
response = client.responses.create(
    model=OPENAI_MODEL,
    instructions="Sos un profesor de programación. Explicá claro y breve.",
    input="Explica qué es una API REST con un ejemplo."
)

print(response.output_text)

Una **API REST** (Representational State Transfer) es una forma de hacer que un sistema se comunique con otro usando **URLs**, **métodos HTTP** y **formato de datos** (generalmente **JSON**).  
REST se basa en que cada “recurso” (por ejemplo: usuarios, productos) se identifica con una URL y se manipula con operaciones estándar.

### Ideas clave
- **Recursos**: cosas como `/users`, `/products`.
- **Métodos HTTP**:
  - **GET**: leer
  - **POST**: crear
  - **PUT/PATCH**: actualizar
  - **DELETE**: borrar
- **Representación**: normalmente se envía/recibe **JSON**.

---

## Ejemplo: API de una tienda

Imaginá estos recursos:

### 1) Obtener una lista de productos
**Request**
- `GET /products`

**Response (JSON)**
```json
[
  { "id": 1, "name": "Mouse", "price": 2500 },
  { "id": 2, "name": "Teclado", "price": 7000 }
]
```

### 2) Obtener un producto específico
**Request**
- `GET /products/2`

**Response (JSON)**
```json
{ "id": 2, "name": "Teclado", "price": 7000 }
```

### 3) Crear un nue

## 5. Mensajes con roles

Útil para separar reglas de la app y pedido del usuario.

In [17]:
response = client.responses.create(
    model=OPENAI_MODEL,
    input=[
        {
            "role": "system",
            "content": "Sos un asistente técnico. Respondé en español, breve y accionable."
        },
        {
            "role": "user",
            "content": "Tengo error 401 al llamar una API. ¿Qué reviso?"
        }
    ]
)

print(response.output_text)

Un **401 Unauthorized** casi siempre es problema de **credenciales/autorizar**, no de “URL” ni de “servidor caído”. Revisá esto en este orden:

1) **Headers de autenticación**
- Asegurate de enviar el header correcto según el API:
  - **Bearer token**: `Authorization: Bearer <token>`
  - **Basic**: `Authorization: Basic base64(user:pass)`
  - **API Key**: a veces va en `x-api-key: ...` (y no en `Authorization`)
- Confirmá que **no estés enviando `Authorization: Bearer null/undefined`**.

2) **Token válido**
- Verificá que el **token no esté expirado**.
- Confirmá que estés usando el **entorno correcto** (dev vs prod) para el token.
- Si usás JWT: revisá `exp` y si el **issuer/audience** coincide (si el backend valida esos campos).

3) **Permisos / scopes / roles**
- Aunque el token sea válido, puede fallar si no tiene el **scope/role** requerido.
- Revisá la documentación del API (ej: `read:users`, `admin`, etc.).

4) **Endpoint / método / firma**
- Algunos APIs exigen:
  - que el toke

## 6. JSON estructurado

Usar Pydantic para recibir datos parseables.

In [ ]:
from typing import Literal
from pydantic import BaseModel


class TicketClasificado(BaseModel):
    categoria: Literal["login", "pagos", "bug", "consulta", "otro"]
    prioridad: Literal["baja", "media", "alta"]
    resumen: str
    respuesta_cliente: str
    comentario_soporte: str


ticket = """
No puedo entrar a mi cuenta desde ayer.
Probé cambiar la contraseña, pero nunca me llega el email.
Necesito acceder urgente porque tengo una factura pendiente.

comentario soporte: Ya lo intentamos. No anduvo.
"""

response = client.responses.parse(
    model=OPENAI_MODEL,
    input=[
        {
            "role": "developer",
            "content": "Clasifica tickets de soporte y genera una respuesta breve. "
        },
        {
            "role": "user",
            "content": ticket
        }
    ],
    text_format=TicketClasificado,
)

resultado = response.output_parsed

print(resultado.model_dump_json(indent=2))

{
  "categoria": "login",
  "prioridad": "alta",
  "resumen": "El cliente no puede acceder a su cuenta desde ayer. Cambió la contraseña, pero no recibe el email de restablecimiento. Necesita acceso urgente por una factura pendiente.",
  "respuesta_cliente": "Entendido. Para resolverlo rápido, verificamos: revisá spam/correos no deseados y que el email registrado sea el correcto. Si sigue sin llegar, generemos un nuevo flujo de recuperación y revisamos el estado de entrega del correo (y/o alternativas de verificación) para que puedas ingresar hoy y ver la factura. Pasanos el email con el que te registraste y, si podés, el número de cliente o último comprobante.",
  "comentario_soporte": "Ya se intentó el cambio/recuperación sin resultado. Requiere revisar email registrado y el envío/estado del restablecimiento (posible bloqueo/baja entrega). Solicitar datos de identificación y confirmar recepción en bandeja/spam."
}


## 7. Ejercicio: extractor de factura

Completar el esquema y probar con texto libre.

In [ ]:
from pydantic import BaseModel


class FacturaExtraida(BaseModel):
    proveedor: str
    monto: float
    moneda: str
    fecha: str
    items: list[str]


texto_factura = """
Factura de Acme SA.
Fecha: 2026-05-10.
Total: USD 1250.50.
Items: hosting anual, soporte premium, dominio.
"""

response = client.responses.parse(
    model=OPENAI_MODEL,
    instructions="Extrae datos de facturas. No inventes campos.",
    input=texto_factura,
    text_format=FacturaExtraida,
)

factura = response.output_parsed # output_text

print(factura.model_dump_json(indent=2))

{
  "proveedor": "Acme SA",
  "monto": 1250.5,
  "moneda": "USD",
  "fecha": "2026-05-10",
  "items": [
    "hosting anual",
    "soporte premium",
    "dominio"
  ]
}


## 8. Function calling

El modelo pide una función.

Python ejecuta la función.

In [23]:
import json


def buscar_pedido(id_pedido: str):
    pedidos = {
        "123": {
            "id": "123",
            "estado": "en camino",
            "fecha_estimada": "2026-05-18",
            "empresa_envio": "Correo Demo"
        },
        "456": {
            "id": "456",
            "estado": "pendiente de pago",
            "fecha_estimada": None,
            "empresa_envio": None
        }
    }

    return pedidos.get(
        id_pedido,
        {
            "id": id_pedido,
            "estado": "no encontrado"
        }
    )


tools = [
    {
        "type": "function",
        "name": "buscar_pedido",
        "description": "Busca el estado de un pedido por ID.",
        "parameters": {
            "type": "object",
            "properties": {
                "id_pedido": {
                    "type": "string",
                    "description": "ID del pedido. Ejemplo: 123."
                }
            },
            "required": ["id_pedido"],
            "additionalProperties": False
        },
        "strict": True
    }
]


input_messages = [
    {
        "role": "user",
        "content": "Hola, quiero saber el estado de mi pedido 123."
    }
]

response = client.responses.create(
    model=OPENAI_MODEL,
    input=input_messages,
    tools=tools
)

print(response)

# output=[ResponseFunctionToolCall(arguments='{"id_pedido":"123"}', call_id='call_tV8EC7K1w03ot6HBeq0IXbBn', name='buscar_pedido', type='function_call', id='fc_02b694c72d3dcaf9006a07b4039f48819fb6491e0f402e1232', namespace=None, status='completed')]

input_messages += response.output


for item in response.output:
    if item.type == "function_call" and item.name == "buscar_pedido":
        args = json.loads(item.arguments)
        resultado = buscar_pedido(id_pedido=args["id_pedido"])

        input_messages.append({
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": json.dumps(resultado, ensure_ascii=False)
        })
    

final_response = client.responses.create(
    model=OPENAI_MODEL,
    input=input_messages,
    tools=tools,
    instructions="Responde al cliente de forma clara y amable."
)

print(final_response.output_text)

Response(id='resp_04e823a1f90a7f79006a07b59b0294819495c3a5272c2dcadc', created_at=1778890139.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-nano-2026-03-17', object='response', output=[ResponseFunctionToolCall(arguments='{"id_pedido":"123"}', call_id='call_9kIo6axFXXoF0DlZjlZn9e2W', name='buscar_pedido', type='function_call', id='fc_04e823a1f90a7f79006a07b59b93b08194b5b2a7c6c065aee1', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='buscar_pedido', parameters={'type': 'object', 'properties': {'id_pedido': {'type': 'string', 'description': 'ID del pedido. Ejemplo: 123.'}}, 'required': ['id_pedido'], 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Busca el estado de un pedido por ID.')], top_p=0.98, background=False, completed_at=1778890139.0, conversation=None, max_output_tokens=None, max_tool_calls=None, previous_response_i

## 9. Web search

Usar cuando la respuesta necesita información actual.

In [24]:
response = client.responses.create(
    model=OPENAI_MODEL,
    tools=[{"type": "web_search"}],
    input="Busca una noticia tecnológica positiva de hoy y resúmela en 3 bullets."
)

print(response.output_text)

- **OpenAI anunció “Daybreak”**, un programa propio de ciberdefensa para **acelerar la seguridad del software** y trabajar con socios de la industria y gobiernos. ([europapress.es](https://www.europapress.es/portaltic/ciberseguridad/noticia-openai-anuncia-propio-programa-ciberdefensa-ejemplo-proyecto-glasswing-anthropic-20260512105840.html?utm_source=openai))  
- La iniciativa **se inspira en “Glasswing” de Anthropic**, que busca identificar vulnerabilidades de **alta gravedad** en sistemas operativos y navegadores web. ([europapress.es](https://www.europapress.es/portaltic/ciberseguridad/noticia-openai-anuncia-propio-programa-ciberdefensa-ejemplo-proyecto-glasswing-anthropic-20260512105840.html?utm_source=openai))  
- La noticia se enmarca como un **paso positivo hacia seguridad más continua y coordinada** en un ecosistema donde el software es crítico (infraestructura y aplicaciones de uso general). ([europapress.es](https://www.europapress.es/portaltic/ciberseguridad/noticia-openai-a